# 23D1 — Dynamic World Setup and Cache Builder
## Utility notebook for NB23D — not a new analysis

### Purpose

NB23D successfully produced:
- exact FIRMS detections,
- 6 robust joint-high candidate cell-days,
- 53 history-novel / environment-plausible comparator cell-days,

but **Dynamic World context was unavailable**.

This notebook does one job only:

> Authenticate Google Earth Engine, extract Dynamic World context for the exact FIRMS points already produced by NB23D, validate observation coverage, and write the cache file that NB23D already knows how to consume.

After this notebook succeeds, rerun NB23D starting from the Dynamic World cache-loading cell.

### Why this is scientifically useful

The output lets us test whether robust hotspot anomalies differ from comparator hotspots in:
- pre-event tree probability,
- crop probability,
- bare-ground probability,
- built probability,
- and same-season year-over-year land-cover probability change.

It also records **valid Sentinel-2 observation counts** so that weak land-cover evidence is not treated as reliable.

### Important interpretation limit

Dynamic World change signals are NOT:
- confirmed deforestation,
- confirmed plantation expansion,
- confirmed land clearing,
- or ignition-cause labels.

## Official source

Dynamic World V1:
`GOOGLE/DYNAMICWORLD/V1`

Google/WRI Dynamic World is a 10 m near-real-time land-cover probability product derived from Sentinel-2.

Official catalog:
https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_DYNAMICWORLD_V1

Earth Engine Python initialization follows the official pattern:

```python
import ee
ee.Authenticate()
ee.Initialize(project="YOUR_PROJECT_ID")
```

You need a Google Cloud / Earth Engine project that your account is allowed to use.

In [2]:
# 1. Imports

from pathlib import Path
import os
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 280)

POINT_BUFFER_M = 200
RECENT_WINDOW_DAYS = 60
MIN_VALID_OBSERVATIONS = 3

In [3]:
# 2. Resolve project root from NB23D output

cwd = Path.cwd()

candidate_roots = [
    cwd,
    cwd.parent,
    Path("/content/skripsi-karhut"),
    Path("/content/drive/MyDrive/skripsi-karhut"),
]

PROJECT_ROOT = None

for root in candidate_roots:
    point_path = (
        root
        / "model_results"
        / "external_context_integration"
        / "t2"
        / "exports"
        / "exact_firms_point_context.csv"
    )

    if point_path.exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate NB23D exact_firms_point_context.csv. "
        "Run NB23D first."
    )

NB23D_DIR = (
    PROJECT_ROOT
    / "model_results"
    / "external_context_integration"
    / "t2"
)

POINT_PATH = (
    NB23D_DIR
    / "exports"
    / "exact_firms_point_context.csv"
)

CACHE_DIR = NB23D_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_PATH = (
    CACHE_DIR
    / "dynamic_world_point_context.csv"
)

DIAGNOSTIC_PATH = (
    CACHE_DIR
    / "dynamic_world_cache_diagnostics.json"
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Input points :", POINT_PATH)
print("Output cache:", CACHE_PATH)

PROJECT_ROOT: c:\Users\Asus\Documents\skripsi-karhut
Input points : c:\Users\Asus\Documents\skripsi-karhut\model_results\external_context_integration\t2\exports\exact_firms_point_context.csv
Output cache: c:\Users\Asus\Documents\skripsi-karhut\model_results\external_context_integration\t2\cache\dynamic_world_point_context.csv


In [4]:
# 3. Load exact FIRMS points generated by NB23D

points = pd.read_csv(
    POINT_PATH,
    parse_dates=["event_date"],
)

required = {
    "firms_point_id",
    "candidate_event_id",
    "context_group",
    "event_date",
    "firms_lat",
    "firms_lon",
}

missing = sorted(required - set(points.columns))
if missing:
    raise ValueError(
        "Missing required exact-point columns:\n"
        + "\n".join(f"- {c}" for c in missing)
    )

points = (
    points[
        list(required)
    ]
    .drop_duplicates("firms_point_id")
    .sort_values(
        ["event_date", "candidate_event_id", "firms_point_id"]
    )
    .reset_index(drop=True)
)

print("Exact FIRMS points:", len(points))
print("Cell-days:", points["candidate_event_id"].nunique())
print("Event dates:", points["event_date"].min().date(), "→", points["event_date"].max().date())

display(points.head())

Exact FIRMS points: 126
Cell-days: 59
Event dates: 2026-08-06 → 2026-08-31


,context_group,firms_lon,firms_point_id,firms_lat,candidate_event_id,event_date
0,History-novel / env-plausible,114.18684,fp_000025,-1.91896,2026-08-06__regional025_10_24,2026-08-06
1,History-novel / env-plausible,115.02630,fp_000046,-1.21862,2026-08-06__regional025_13_28,2026-08-06
2,History-novel / env-plausible,115.73869,fp_000066,-3.62061,2026-08-06__regional025_3_30,2026-08-06
3,History-novel / env-plausible,115.57914,fp_000068,-3.34097,2026-08-06__regional025_4_30,2026-08-06
4,History-novel / env-plausible,113.60196,fp_000075,-2.97716,2026-08-06__regional025_6_22,2026-08-06


# PART A — Install / authenticate Earth Engine

If `earthengine-api` is not installed, run:

```bash
pip install earthengine-api
```

Then restart the kernel if necessary.

For local Jupyter, `ee.Authenticate()` normally opens or provides a browser-based Google authorization flow.

Use the SAME Google account that has access to your Earth Engine project.

In [5]:
# 4. Import Earth Engine

try:
    import ee
except ImportError as exc:
    raise ImportError(
        "earthengine-api is not installed. Run:\n"
        "pip install earthengine-api\n"
        "Then restart the kernel and rerun this notebook."
    ) from exc

print("earthengine-api imported.")

earthengine-api imported.


In [6]:
# 5. Set your Earth Engine / Google Cloud project

# Project ID yang barusan dibuat:
MANUAL_EE_PROJECT = "skripsi-karhut"

# Optional environment variable override.
EE_PROJECT = os.getenv(
    "EE_PROJECT",
    "",
).strip()

if not EE_PROJECT:
    EE_PROJECT = MANUAL_EE_PROJECT.strip()

print("EE_PROJECT:", EE_PROJECT if EE_PROJECT else "<not set>")

if not EE_PROJECT:
    raise ValueError(
        "EE_PROJECT masih kosong. Isi MANUAL_EE_PROJECT dengan Project ID Google Cloud/Earth Engine kamu."
    )


EE_PROJECT: skripsi-karhut


In [7]:
# 6. Authenticate + initialize
#
# Default untuk Jupyter yang berjalan lokal di laptop:
#   auth_mode="localhost"
#
# Jika localhost gagal karena environment/browser tertentu,
# notebook otomatis mencoba auth_mode="notebook".
#
# PENTING:
# - Jangan kirim authorization code/token ke siapa pun.
# - Paste code hanya ke prompt notebook jika fallback "notebook" digunakan.

if not EE_PROJECT:
    raise ValueError(
        "EE_PROJECT kosong. Isi MANUAL_EE_PROJECT di cell sebelumnya."
    )

def initialize_earth_engine(project_id):
    # First try existing credentials.
    try:
        ee.Initialize(project=project_id)
        print("Earth Engine already authenticated and initialized.")
        return
    except Exception:
        print("Belum ada credential aktif untuk project ini.")
        print("Mencoba autentikasi mode localhost...")

    # Preferred mode for local Jupyter on the same computer as the browser.
    try:
        ee.Authenticate(
            auth_mode="localhost",
            force=True,
        )
        ee.Initialize(project=project_id)
        print("Earth Engine authenticated with localhost mode.")
        return
    except Exception as localhost_exc:
        print()
        print("Mode localhost gagal.")
        print("Detail:", repr(localhost_exc))
        print()
        print("Mencoba fallback auth_mode='notebook'...")
        print(
            "Jika browser memberikan authorization code, "
            "paste LANGSUNG ke prompt di notebook ini."
        )

    # Fallback for Jupyter environments where localhost callback cannot work.
    ee.Authenticate(
        auth_mode="notebook",
        force=True,
    )
    ee.Initialize(project=project_id)
    print("Earth Engine authenticated with notebook mode.")


initialize_earth_engine(EE_PROJECT)

# Minimal server-side connection test.
test_value = ee.Number(1).getInfo()
assert test_value == 1

print()
print("Earth Engine connection test: OK")
print("Project:", EE_PROJECT)


Belum ada credential aktif untuk project ini.
Mencoba autentikasi mode localhost...



Successfully saved authorization token.


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine authenticated with localhost mode.

Earth Engine connection test: OK
Project: skripsi-karhut


## Jika autentikasi masih gagal

### Kasus A — Browser terbuka lalu langsung selesai
Itu normal untuk `auth_mode="localhost"`. Kembali ke notebook dan tunggu cell selesai.

### Kasus B — Browser memberikan authorization code
Berarti fallback `auth_mode="notebook"` sedang dipakai.  
Copy code tersebut dan paste **langsung ke prompt notebook**.

Jangan screenshot atau mengirim authorization code/token.

### Kasus C — `PermissionDenied` / project tidak terdaftar
Pastikan:
1. Project ID adalah `skripsi-karhut`.
2. Earth Engine API aktif.
3. Project sudah diregister untuk Earth Engine noncommercial.
4. Akun Google yang dipakai untuk autentikasi sama dengan akun yang punya akses project.

### Kasus D — `earthengine-api` belum ada
Jalankan di terminal / Anaconda Prompt:

```bash
pip install -U earthengine-api
```

Lalu restart kernel Jupyter dan Run All lagi.

In [8]:
# 7. Verify Dynamic World availability over the thesis period

dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")

thesis_window = (
    dw
    .filterDate("2026-06-01", "2026-09-01")
)

n_images = thesis_window.size().getInfo()

print("Dynamic World images in Jun-Aug 2026:", n_images)

if n_images == 0:
    raise RuntimeError(
        "No Dynamic World imagery returned for the thesis period."
    )

Dynamic World images in Jun-Aug 2026: 667225


# PART B — Dynamic World extraction design

For each exact FIRMS detection and event date:

### Recent pre-event period
`event_date - 60 days` up to but excluding `event_date`

### Reference period
The same relative 60-day window one year earlier.

For each period we calculate mean probability for:
- water
- trees
- grass
- flooded vegetation
- crops
- shrub/scrub
- built
- bare
- snow/ice

We also calculate:

- `dw_recent_valid_obs`
- `dw_ref_valid_obs`

These are valid Dynamic World observation counts at the point buffer.

### Reliability rule

`dw_context_reliable = True`

only when both recent and reference periods have at least **3 valid observations**.

This threshold is a pragmatic data-quality gate, not a physical law.

In [9]:
# 8. Feature definitions

DW_BANDS = [
    "water",
    "trees",
    "grass",
    "flooded_vegetation",
    "crops",
    "shrub_and_scrub",
    "built",
    "bare",
    "snow_and_ice",
]

print("Dynamic World bands:", DW_BANDS)

Dynamic World bands: ['water', 'trees', 'grass', 'flooded_vegetation', 'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice']


In [10]:
# 9. Build one event-date Dynamic World image stack

def build_dw_stack(event_date):
    event_date = pd.Timestamp(event_date)

    recent_start = (
        event_date
        - pd.Timedelta(days=RECENT_WINDOW_DAYS)
    )

    recent_end = event_date

    ref_start = (
        recent_start
        - pd.DateOffset(years=1)
    )

    ref_end = (
        recent_end
        - pd.DateOffset(years=1)
    )

    recent_col = (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterDate(
            str(recent_start.date()),
            str(recent_end.date()),
        )
    )

    ref_col = (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterDate(
            str(ref_start.date()),
            str(ref_end.date()),
        )
    )

    recent_mean = (
        recent_col
        .select(DW_BANDS)
        .mean()
        .rename(
            [f"dw_recent_{b}" for b in DW_BANDS]
        )
    )

    ref_mean = (
        ref_col
        .select(DW_BANDS)
        .mean()
        .rename(
            [f"dw_ref_{b}" for b in DW_BANDS]
        )
    )

    delta = (
        recent_mean
        .select(
            [f"dw_recent_{b}" for b in DW_BANDS]
        )
        .subtract(
            ref_mean.select(
                [f"dw_ref_{b}" for b in DW_BANDS]
            )
        )
        .rename(
            [f"dw_delta_{b}" for b in DW_BANDS]
        )
    )

    # Valid observation counts. Use trees only as the count proxy because all
    # Dynamic World probability bands share the same image mask.
    recent_count = (
        recent_col
        .select(["trees"])
        .count()
        .rename(["dw_recent_valid_obs"])
    )

    ref_count = (
        ref_col
        .select(["trees"])
        .count()
        .rename(["dw_ref_valid_obs"])
    )

    stack = (
        recent_mean
        .addBands(ref_mean)
        .addBands(delta)
        .addBands(recent_count)
        .addBands(ref_count)
    )

    metadata = {
        "recent_start": str(recent_start.date()),
        "recent_end_exclusive": str(recent_end.date()),
        "reference_start": str(ref_start.date()),
        "reference_end_exclusive": str(ref_end.date()),
        "recent_collection_size": int(recent_col.size().getInfo()),
        "reference_collection_size": int(ref_col.size().getInfo()),
    }

    return stack, metadata

In [11]:
# 10. Extract Dynamic World context for one event date

def extract_for_event_date(points_for_date, event_date):
    stack, metadata = build_dw_stack(event_date)

    features = []

    for row in points_for_date.itertuples(index=False):
        geometry = (
            ee.Geometry.Point(
                [
                    float(row.firms_lon),
                    float(row.firms_lat),
                ]
            )
            .buffer(POINT_BUFFER_M)
        )

        feature = ee.Feature(
            geometry,
            {
                "firms_point_id": row.firms_point_id,
                "candidate_event_id": row.candidate_event_id,
                "context_group": row.context_group,
                "event_date": str(
                    pd.Timestamp(row.event_date).date()
                ),
            },
        )

        features.append(feature)

    fc = ee.FeatureCollection(features)

    reduced = stack.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.mean(),
        scale=10,
        tileScale=4,
    )

    info = reduced.getInfo()

    rows = []

    for feature in info.get("features", []):
        props = feature.get("properties", {})
        rows.append(props)

    result = pd.DataFrame(rows)

    for key, value in metadata.items():
        result[key] = value

    return result

In [12]:
# 11. Run extraction date by date

parts = []

date_metadata = []

for event_date, subset in points.groupby("event_date"):
    event_date = pd.Timestamp(event_date)

    print(
        "Extracting:",
        event_date.date(),
        "| points:",
        len(subset),
    )

    part = extract_for_event_date(
        subset,
        event_date,
    )

    if len(part):
        parts.append(part)

        date_metadata.append({
            "event_date": str(event_date.date()),
            "points_requested": int(len(subset)),
            "points_returned": int(len(part)),
            "recent_collection_size": (
                int(part["recent_collection_size"].iloc[0])
                if "recent_collection_size" in part.columns
                else None
            ),
            "reference_collection_size": (
                int(part["reference_collection_size"].iloc[0])
                if "reference_collection_size" in part.columns
                else None
            ),
        })

if not parts:
    raise RuntimeError(
        "Dynamic World extraction returned zero rows."
    )

dw_context = pd.concat(
    parts,
    ignore_index=True,
)

print()
print("Rows returned:", len(dw_context))

Extracting: 2026-08-06 | points: 12
Extracting: 2026-08-07 | points: 6
Extracting: 2026-08-08 | points: 12
Extracting: 2026-08-09 | points: 2
Extracting: 2026-08-10 | points: 8
Extracting: 2026-08-11 | points: 2
Extracting: 2026-08-12 | points: 3
Extracting: 2026-08-14 | points: 4
Extracting: 2026-08-15 | points: 7
Extracting: 2026-08-16 | points: 6
Extracting: 2026-08-17 | points: 19
Extracting: 2026-08-18 | points: 10
Extracting: 2026-08-19 | points: 3
Extracting: 2026-08-22 | points: 3
Extracting: 2026-08-23 | points: 2
Extracting: 2026-08-25 | points: 1
Extracting: 2026-08-26 | points: 11
Extracting: 2026-08-27 | points: 3
Extracting: 2026-08-28 | points: 4
Extracting: 2026-08-29 | points: 6
Extracting: 2026-08-31 | points: 2

Rows returned: 126


In [13]:
# 12. Normalize numeric columns and derive quality flags

numeric_cols = [
    c for c in dw_context.columns
    if (
        c.startswith("dw_recent_")
        or c.startswith("dw_ref_")
        or c.startswith("dw_delta_")
    )
]

for col in numeric_cols:
    dw_context[col] = pd.to_numeric(
        dw_context[col],
        errors="coerce",
    )

dw_context["dw_context_reliable"] = (
    dw_context["dw_recent_valid_obs"].ge(
        MIN_VALID_OBSERVATIONS
    )
    &
    dw_context["dw_ref_valid_obs"].ge(
        MIN_VALID_OBSERVATIONS
    )
)

# Dominant recent class based on mean probability.
recent_prob_cols = [
    f"dw_recent_{b}"
    for b in DW_BANDS
]

def dominant_class(row):
    values = row[recent_prob_cols]

    if values.isna().all():
        return np.nan

    best_col = values.astype(float).idxmax()
    return best_col.replace("dw_recent_", "")

def dominant_confidence(row):
    values = row[recent_prob_cols]

    if values.isna().all():
        return np.nan

    return float(values.astype(float).max())

dw_context["dw_recent_dominant_class"] = (
    dw_context.apply(
        dominant_class,
        axis=1,
    )
)

dw_context["dw_recent_dominant_probability"] = (
    dw_context.apply(
        dominant_confidence,
        axis=1,
    )
)

print(
    "Reliable context coverage:",
    round(
        dw_context["dw_context_reliable"].mean(),
        4,
    ),
)

display(
    dw_context[
        [
            "firms_point_id",
            "candidate_event_id",
            "context_group",
            "event_date",
            "dw_recent_valid_obs",
            "dw_ref_valid_obs",
            "dw_context_reliable",
            "dw_recent_dominant_class",
            "dw_recent_dominant_probability",
            "dw_recent_trees",
            "dw_recent_crops",
            "dw_recent_bare",
            "dw_delta_trees",
            "dw_delta_crops",
            "dw_delta_bare",
        ]
    ].head(30)
)

Reliable context coverage: 0.2778


,firms_point_id,candidate_event_id,context_group,event_date,dw_recent_valid_obs,dw_ref_valid_obs,dw_context_reliable,dw_recent_dominant_class,dw_recent_dominant_probability,dw_recent_trees,dw_recent_crops,dw_recent_bare,dw_delta_trees,dw_delta_crops,dw_delta_bare
0,fp_000025,2026-08-06__regional025_10_24,History-novel / env-plausible,2026-08-06,1.000000,2.000000,False,trees,0.684154,0.684154,0.032512,0.037587,-0.008008,0.000116,0.004153
1,fp_000046,2026-08-06__regional025_13_28,History-novel / env-plausible,2026-08-06,1.000000,1.182421,False,trees,0.675609,0.675609,0.030307,0.043960,-0.055143,-0.000098,0.009351
2,fp_000066,2026-08-06__regional025_3_30,History-novel / env-plausible,2026-08-06,3.839800,1.152371,False,trees,0.566284,0.566284,0.050734,0.048770,-0.068039,0.011314,0.009929
3,fp_000068,2026-08-06__regional025_4_30,History-novel / env-plausible,2026-08-06,1.000000,NaN,False,trees,0.680326,0.680326,0.042200,0.034262,NaN,NaN,NaN
4,fp_000075,2026-08-06__regional025_6_22,History-novel / env-plausible,2026-08-06,2.000000,1.037768,False,trees,0.674262,0.674262,0.027543,0.033758,0.032456,0.002272,-0.000878
5,fp_000080,2026-08-06__regional025_6_23,History-novel / env-plausible,2026-08-06,3.196774,2.000000,False,trees,0.622005,0.622005,0.032089,0.035164,0.062573,-0.009193,-0.017443
6,fp_000089,2026-08-06__regional025_6_27,History-novel / env-plausible,2026-08-06,5.000000,5.091266,True,trees,0.535781,0.535781,0.039771,0.032629,0.190049,-0.004930,-0.014742
7,fp_000090,2026-08-06__regional025_6_27,History-novel / env-plausible,2026-08-06,5.472917,5.363226,True,trees,0.669735,0.669735,0.034433,0.033014,0.088849,0.000772,-0.003320
8,fp_000113,2026-08-06__regional025_9_24,History-novel / env-plausible,2026-08-06,2.426291,2.000000,False,trees,0.730689,0.730689,0.029573,0.036200,-0.000532,-0.000977,0.001483
9,fp_000114,2026-08-06__regional025_9_24,History-novel / env-plausible,2026-08-06,2.000000,2.744326,False,trees,0.731482,0.731482,0.029612,0.035631,-0.003096,0.000420,0.000537


In [14]:
# 13. Coverage diagnostics by analysis group

coverage_summary = (
    dw_context
    .groupby("context_group")
    .agg(
        points=("firms_point_id", "size"),
        reliable_points=("dw_context_reliable", "sum"),
        recent_obs_median=("dw_recent_valid_obs", "median"),
        reference_obs_median=("dw_ref_valid_obs", "median"),
        dominant_probability_median=(
            "dw_recent_dominant_probability",
            "median",
        ),
    )
    .reset_index()
)

coverage_summary["reliable_share"] = (
    coverage_summary["reliable_points"]
    / coverage_summary["points"]
)

display(coverage_summary)

,context_group,points,reliable_points,recent_obs_median,reference_obs_median,dominant_probability_median,reliable_share
0,History-novel / env-plausible,111,35,4.066470,2.208986,0.653747,0.315315
1,Robust joint-high,15,0,1.087108,1.000000,0.587037,0.000000


In [15]:
# 14. Validate one-to-one point coverage

input_ids = set(points["firms_point_id"])
output_ids = set(dw_context["firms_point_id"])

missing_ids = sorted(input_ids - output_ids)
extra_ids = sorted(output_ids - input_ids)

print("Input point IDs :", len(input_ids))
print("Output point IDs:", len(output_ids))
print("Missing IDs     :", len(missing_ids))
print("Unexpected IDs  :", len(extra_ids))

if missing_ids:
    print("First missing IDs:", missing_ids[:20])

if extra_ids:
    print("First unexpected IDs:", extra_ids[:20])

if extra_ids:
    raise RuntimeError(
        "Dynamic World returned unexpected FIRMS point IDs."
    )

Input point IDs : 126
Output point IDs: 126
Missing IDs     : 0
Unexpected IDs  : 0


In [16]:
# 15. Save cache in the exact path expected by NB23D

# Keep the columns NB23D expects, plus diagnostics.
dw_context.to_csv(
    CACHE_PATH,
    index=False,
)

diagnostics = {
    "input_point_rows": int(len(points)),
    "output_rows": int(len(dw_context)),
    "unique_input_point_ids": int(len(input_ids)),
    "unique_output_point_ids": int(len(output_ids)),
    "missing_point_ids": missing_ids,
    "unexpected_point_ids": extra_ids,
    "point_buffer_m": POINT_BUFFER_M,
    "recent_window_days": RECENT_WINDOW_DAYS,
    "minimum_valid_observations": MIN_VALID_OBSERVATIONS,
    "reliable_share": float(
        dw_context["dw_context_reliable"].mean()
    ),
    "date_metadata": date_metadata,
}

DIAGNOSTIC_PATH.write_text(
    json.dumps(
        diagnostics,
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved Dynamic World cache:")
print(CACHE_PATH)

print()
print("Saved diagnostics:")
print(DIAGNOSTIC_PATH)

Saved Dynamic World cache:
c:\Users\Asus\Documents\skripsi-karhut\model_results\external_context_integration\t2\cache\dynamic_world_point_context.csv

Saved diagnostics:
c:\Users\Asus\Documents\skripsi-karhut\model_results\external_context_integration\t2\cache\dynamic_world_cache_diagnostics.json


# PART C — Immediate sanity checks before rerunning NB23D

The cache should NOT be trusted simply because the extraction completed.

Check:

1. `Reliable context coverage` should be reasonably high.
2. Recent/reference observation counts should not be mostly 0–1.
3. Dominant probabilities should not all be extremely low.
4. Candidate and comparator groups should both have usable coverage.
5. No major point-ID mismatch should exist.

If coverage is poor, we change the compositing window before doing any interpretation.

In [17]:
# 16. Simple data-quality gate

overall_reliable_share = float(
    dw_context["dw_context_reliable"].mean()
)

candidate_coverage = (
    dw_context
    .groupby("context_group")[
        "dw_context_reliable"
    ]
    .mean()
)

print(
    "Overall reliable share:",
    round(overall_reliable_share, 4),
)

display(
    candidate_coverage
    .rename("reliable_share")
    .to_frame()
)

print()

if overall_reliable_share >= 0.80:
    print(
        "PASS: Dynamic World coverage is strong enough for the planned "
        "candidate-vs-comparator contextual analysis."
    )

elif overall_reliable_share >= 0.60:
    print(
        "CAUTION: Dynamic World coverage is usable but incomplete. "
        "Interpret group comparisons with explicit coverage reporting."
    )

else:
    print(
        "STOP: Dynamic World coverage is too weak for reliable comparison. "
        "Do not interpret land-cover differences yet."
    )

Overall reliable share: 0.2778


,reliable_share
context_group,
History-novel / env-plausible,0.315315
Robust joint-high,0.000000



STOP: Dynamic World coverage is too weak for reliable comparison. Do not interpret land-cover differences yet.


# Next action

If the final quality gate says **PASS** or **CAUTION**:

1. Close/save this notebook.
2. Open `23D_external_context_data_integration.ipynb`.
3. Rerun from the cell titled:
   **"12. Run Dynamic World extraction"**

Because the cache now exists, NB23D should print:

`Loaded cached Dynamic World context: ...`

Then the source availability table should change from:

`Dynamic World = False`

to:

`Dynamic World = True`

Send the newly executed NB23D back for review.

Do NOT proceed to GeoMAE yet.  
First we evaluate whether Dynamic World actually separates the 6 robust candidates from the comparator group.